## Recursive JSON Splitter

`RecursiveJsonSplitter` is a LangChain tool that splits large JSON data into smaller chunks while preserving its nested structure.

### How it works

1. Takes JSON data as a Python dictionary.
2. Recursively explores nested objects.
3. Splits them into smaller chunks while keeping parent keys for context.

### Key options

- **`max_chunk_size`**: Target maximum chunk size in characters, not tokens.
- **`min_chunk_size`**: Guides when to start a new chunk; it is not a guaranteed minimum.
- **`convert_lists=True`**: Converts lists into dictionaries with index keys so their items can be split.

### Available methods

- **`split_json()`**: Returns chunks as Python dictionaries.
- **`split_text()`**: Returns chunks as JSON-formatted strings.
- **`create_documents()`**: Returns LangChain `Document` objects.

### Example

```python
from langchain_text_splitters import RecursiveJsonSplitter

json_data = {
    "course": {
        "name": "Artificial Intelligence",
        "topics": {
            "machine_learning": "Learning patterns from data.",
            "deep_learning": "Learning using neural networks.",
            "nlp": "Understanding and processing human language."
        }
    }
}

splitter = RecursiveJsonSplitter(
    max_chunk_size=120,
    min_chunk_size=50
)

chunks = splitter.split_json(json_data=json_data)

for chunk in chunks:
    print(chunk)
```

### Limitation

Long individual strings are not split, and lists remain intact by default. Therefore, some chunks may exceed `max_chunk_size`.

### Use cases

- Processing large API responses.
- Preparing structured JSON data for RAG.
- Keeping nested context available during document retrieval.

In [1]:
import json
import requests

json_data=requests.get("https://api.smith.langchain.com/openapi.json").json()

In [3]:
json_data_raw = json.dumps(json_data, indent=4)  # Convert to a formatted JSON string
json_data_raw

'{\n    "openapi": "3.1.0",\n    "info": {\n        "title": "LangSmith",\n        "description": "The LangSmith API is used to programmatically create and manage LangSmith resources.\\n\\n## Host\\nhttps://api.smith.langchain.com\\n\\n## Authentication\\nTo authenticate with the LangSmith API, set the `X-Api-Key` header\\nto a valid [LangSmith API key](https://docs.langchain.com/langsmith/create-account-api-key#create-an-api-key).\\n\\n",\n        "version": "0.1.0"\n    },\n    "paths": {\n        "/api/v1/info/health": {\n            "get": {\n                "tags": [\n                    "info"\n                ],\n                "summary": "Get Health Info",\n                "description": "Get health information about the current deployment of LangSmith.",\n                "operationId": "get_health_info_api_v1_info_health_get",\n                "responses": {\n                    "200": {\n                        "description": "Successful Response",\n                        "

In [6]:
from langchain_text_splitters import RecursiveJsonSplitter

json_splitter = RecursiveJsonSplitter(max_chunk_size=1000)
json_chunks = json_splitter.split_text(json_data)
json_chunks

['{"openapi": "3.1.0", "info": {"title": "LangSmith", "description": "The LangSmith API is used to programmatically create and manage LangSmith resources.\\n\\n## Host\\nhttps://api.smith.langchain.com\\n\\n## Authentication\\nTo authenticate with the LangSmith API, set the `X-Api-Key` header\\nto a valid [LangSmith API key](https://docs.langchain.com/langsmith/create-account-api-key#create-an-api-key).\\n\\n", "version": "0.1.0"}, "paths": {"/api/v1/info/health": {"get": {"tags": ["info"], "summary": "Get Health Info", "description": "Get health information about the current deployment of LangSmith.", "operationId": "get_health_info_api_v1_info_health_get", "responses": {"200": {"description": "Successful Response", "content": {"application/json": {"schema": {"$ref": "#/components/schemas/HealthInfoGetResponse"}}}}}, "x-public": true}}}}',
 '{"paths": {"/api/v1/sessions/{session_id}/dashboard": {"post": {"tags": ["tracer-sessions"], "summary": "Get Tracing Project Prebuilt Dashboard",

In [9]:
for chunk in json_chunks[:3]:
    print(chunk)

{"openapi": "3.1.0", "info": {"title": "LangSmith", "description": "The LangSmith API is used to programmatically create and manage LangSmith resources.\n\n## Host\nhttps://api.smith.langchain.com\n\n## Authentication\nTo authenticate with the LangSmith API, set the `X-Api-Key` header\nto a valid [LangSmith API key](https://docs.langchain.com/langsmith/create-account-api-key#create-an-api-key).\n\n", "version": "0.1.0"}, "paths": {"/api/v1/info/health": {"get": {"tags": ["info"], "summary": "Get Health Info", "description": "Get health information about the current deployment of LangSmith.", "operationId": "get_health_info_api_v1_info_health_get", "responses": {"200": {"description": "Successful Response", "content": {"application/json": {"schema": {"$ref": "#/components/schemas/HealthInfoGetResponse"}}}}}, "x-public": true}}}}
{"paths": {"/api/v1/sessions/{session_id}/dashboard": {"post": {"tags": ["tracer-sessions"], "summary": "Get Tracing Project Prebuilt Dashboard", "description":

In [14]:
# The splitter can also output documents

json_docs = json_splitter.create_documents(texts=[json_data])
json_docs


[Document(metadata={}, page_content='{"openapi": "3.1.0", "info": {"title": "LangSmith", "description": "The LangSmith API is used to programmatically create and manage LangSmith resources.\\n\\n## Host\\nhttps://api.smith.langchain.com\\n\\n## Authentication\\nTo authenticate with the LangSmith API, set the `X-Api-Key` header\\nto a valid [LangSmith API key](https://docs.langchain.com/langsmith/create-account-api-key#create-an-api-key).\\n\\n", "version": "0.1.0"}, "paths": {"/api/v1/info/health": {"get": {"tags": ["info"], "summary": "Get Health Info", "description": "Get health information about the current deployment of LangSmith.", "operationId": "get_health_info_api_v1_info_health_get", "responses": {"200": {"description": "Successful Response", "content": {"application/json": {"schema": {"$ref": "#/components/schemas/HealthInfoGetResponse"}}}}}, "x-public": true}}}}'),
 Document(metadata={}, page_content='{"paths": {"/api/v1/sessions/{session_id}/dashboard": {"post": {"tags": ["

In [15]:
for chunk in json_docs[:3]:
    print(chunk)

page_content='{"openapi": "3.1.0", "info": {"title": "LangSmith", "description": "The LangSmith API is used to programmatically create and manage LangSmith resources.\n\n## Host\nhttps://api.smith.langchain.com\n\n## Authentication\nTo authenticate with the LangSmith API, set the `X-Api-Key` header\nto a valid [LangSmith API key](https://docs.langchain.com/langsmith/create-account-api-key#create-an-api-key).\n\n", "version": "0.1.0"}, "paths": {"/api/v1/info/health": {"get": {"tags": ["info"], "summary": "Get Health Info", "description": "Get health information about the current deployment of LangSmith.", "operationId": "get_health_info_api_v1_info_health_get", "responses": {"200": {"description": "Successful Response", "content": {"application/json": {"schema": {"$ref": "#/components/schemas/HealthInfoGetResponse"}}}}}, "x-public": true}}}}'
page_content='{"paths": {"/api/v1/sessions/{session_id}/dashboard": {"post": {"tags": ["tracer-sessions"], "summary": "Get Tracing Project Prebui

In [16]:
for chunk in json_docs[:3]:
    print("Content:", chunk.page_content)
    print("Metadata:", chunk.metadata)
    print()

Content: {"openapi": "3.1.0", "info": {"title": "LangSmith", "description": "The LangSmith API is used to programmatically create and manage LangSmith resources.\n\n## Host\nhttps://api.smith.langchain.com\n\n## Authentication\nTo authenticate with the LangSmith API, set the `X-Api-Key` header\nto a valid [LangSmith API key](https://docs.langchain.com/langsmith/create-account-api-key#create-an-api-key).\n\n", "version": "0.1.0"}, "paths": {"/api/v1/info/health": {"get": {"tags": ["info"], "summary": "Get Health Info", "description": "Get health information about the current deployment of LangSmith.", "operationId": "get_health_info_api_v1_info_health_get", "responses": {"200": {"description": "Successful Response", "content": {"application/json": {"schema": {"$ref": "#/components/schemas/HealthInfoGetResponse"}}}}}, "x-public": true}}}}
Metadata: {}

Content: {"paths": {"/api/v1/sessions/{session_id}/dashboard": {"post": {"tags": ["tracer-sessions"], "summary": "Get Tracing Project Pre

In [19]:
# Get the string documents from the splitter

json_string_docs = json_splitter.split_text(json_data)
print("Number of string documents:", len(json_string_docs))
print("First string document:", json_string_docs[0])
print("Second string document:", json_string_docs[1])

Number of string documents: 1337
First string document: {"openapi": "3.1.0", "info": {"title": "LangSmith", "description": "The LangSmith API is used to programmatically create and manage LangSmith resources.\n\n## Host\nhttps://api.smith.langchain.com\n\n## Authentication\nTo authenticate with the LangSmith API, set the `X-Api-Key` header\nto a valid [LangSmith API key](https://docs.langchain.com/langsmith/create-account-api-key#create-an-api-key).\n\n", "version": "0.1.0"}, "paths": {"/api/v1/info/health": {"get": {"tags": ["info"], "summary": "Get Health Info", "description": "Get health information about the current deployment of LangSmith.", "operationId": "get_health_info_api_v1_info_health_get", "responses": {"200": {"description": "Successful Response", "content": {"application/json": {"schema": {"$ref": "#/components/schemas/HealthInfoGetResponse"}}}}}, "x-public": true}}}}
Second string document: {"paths": {"/api/v1/sessions/{session_id}/dashboard": {"post": {"tags": ["tracer